In [ ]:
def print_2d_matrix(lst, raw=False):
    print(*[num for num in lst], sep="\n")
    return
    print(*[round(num, 8) for num in lst], sep="\n")

In [ ]:
import numpy as np

q = ["cold", "hot"]
N = len(q)

# a[i][j] probability of moving from state i to state j
a = np.random.dirichlet(np.ones(N), size=N)

# All of the observations available
v = ["1", "2", "3"]

# b[i][j] probability of getting observation j knowing state i happened
b = [
    dict(zip(v, np.random.dirichlet(np.ones(len(v)))))
    for _ in range(N)
]

pi = np.squeeze(np.random.dirichlet(np.ones(N), size=1))

# print_2d_matrix(a)
# print_2d_matrix(b, raw=True)
# print_2d_matrix(pi)
print(a)
print()
print(b)
print()
print(pi)

[[0.99467612 0.00532388]
 [0.95864334 0.04135666]]

[{'1': np.float64(0.12019206780629914), '2': np.float64(0.2985383098889895), '3': np.float64(0.5812696223047114)}, {'1': np.float64(0.48165337839243527), '2': np.float64(0.34577831347560006), '3': np.float64(0.17256830813196453)}]

[0.82619693 0.17380307]


In [ ]:
o = ["2", "1", "1", "3"]
T = len(o)

alpha = np.zeros((N, T), dtype=float)
beta = np.zeros((N, T), dtype=float)

# gamma = [[0 for _ in range(T)] for _ in range(N)]


In [ ]:
def update_alpha():
    global alpha
    alpha = np.zeros((N, T), dtype=float)
    for state in range(N):
        alpha[state, 0] = pi[state] * b[state][o[0]]

    for t in range(1, T):
        observation = o[t]
        for state in range(N):
            for state_prev in range(N):
                alpha[state][t] += alpha[state_prev][t - 1] * a[state_prev][state] * b[state][observation]

update_alpha()
print(alpha)

[[2.46651434e-01 3.64122111e-02 4.56396729e-03 2.71110882e-03]
 [6.00973332e-02 1.82959253e-03 1.29815215e-04 5.11953698e-06]]


In [ ]:
def update_beta():
    global beta
    beta = np.zeros((N, T), dtype=float)
    beta[:, -1] = 1.0
    for state in range(N):
        beta[state, -1] = pi[state] * b[state][o[0]]

    for t in range(T-2, -1, -1):
        observation = o[t + 1]
        for state in range(N):
            for state_next in range(N):
                beta[state][t] += beta[state_next][t + 1] * a[state][state_next] * b[state_next][observation]

update_beta()
print(beta)

[[0.0021305  0.0174092  0.14266291 0.24665143]
 [0.00238805 0.01918413 0.13787055 0.06009733]]


In [ ]:
# The probability of being in state i at time t and j at time t+1
def xi(t, i, j):
    not_quite_xi = alpha[i, t] * a[i, j] * b[j][o[t + 1]] * beta[j, t+1]
    denominator = np.sum(alpha[:, t] * beta[:, t])
    return not_quite_xi / denominator

In [ ]:
# The probability of being in state j at time t
def gamma(t, j):
    numerator = alpha[j, t] * beta[j, t]
    denominator = np.sum(alpha[:, t] * beta[:, t])
    return numerator / denominator

In [ ]:
def update_a_at(i, j):
    global a
    a[i, j] = sum(xi(t, i, j) for t in range(T-1))/sum(sum(xi(t, i, k) for k in range(N)) for t in range(T-1))

def update_a():
    for i in range(N):
        for j in range(N):
            update_a_at(i, j)

In [ ]:
def update_b_at(j, v_k):
    global b
    numerator = sum(gamma(t, j) for t in range(T) if o[t] == v_k)
    denominator = sum(gamma(t, j) for t in range(T))
    b[j][v_k] = numerator / denominator

def update_b():
    for j in range(N):
        for v_k in o:
            update_b_at(j, v_k)

In [ ]:
def update_pi():
    global pi

    for i in range(N):
        pi[i] = gamma(0, i)

0.0032


1


[1, 0, 0, 1]


hot cold cold hot 